# Vietnam Real Estate Valuation & Conformal Prediction — Kaggle Full Option A Runner

Chạy toàn bộ pipeline 4 thành viên (TV1–TV4) với cấu hình `configs/kaggle_full.yaml` trên GPU Kaggle T4/P100.

Bật Internet và GPU. Có thể cung cấp checkout qua VN_RE_PROJECT_ROOT.

In [ ]:
import json
import os
import subprocess
import sys
import tarfile
from pathlib import Path

repo = os.environ.get('VN_RE_REPO_URL', 'https://github.com/DungNguyen2203/CS114-Nhom-8.git')
branch = os.environ.get('VN_RE_BRANCH', 'feat/full-pipeline')
root = Path(os.environ.get('VN_RE_PROJECT_ROOT', '/kaggle/working/CS114-Nhom-8')).resolve()
if not (root / 'configs').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, repo, str(root)], check=True)
os.chdir(root)
os.environ.update(VN_RE_PROJECT_ROOT=str(root), HF_HOME='/kaggle/working/hf_cache', TOKENIZERS_PARALLELISM='false')
revision = os.environ.get('VN_RE_COMMIT')
if revision:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', revision], check=True)
    subprocess.run(['git', 'checkout', '--detach', revision], check=True)
commit = None
if (root / '.git').exists():
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    print(f'Checkout {commit}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '.[dev]'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-v'], check=True)
for stage in ('prepare_data', 'extract_embeddings', 'train_eval', 'build_reports'):
    subprocess.run([sys.executable, '-m', f'src.pipelines.{stage}', '--config', 'configs/kaggle_full.yaml'], check=True)

summary = json.loads(Path('outputs/metrics/run_summary.json').read_text(encoding='utf-8'))
files = [Path('outputs') / name for name in summary['files']['outputs_dir']]
files += [Path('models') / name for name in summary['files']['models_dir']]
files += [Path('outputs/metrics/run_summary.json'), Path('data/processed/splits/split_manifest.json')]
files += [Path('outputs/audits') / name for name in ('audit_labels_300.csv', 'audit_masking_100.csv')]
files += [Path('reports') / name for name in ('data_report.md', 'data_dictionary.md', 'experiment_report.md', 'slide_outline.md')]
files += [Path('outputs/figures') / name for name in ('fig_01_funnel.png', 'fig_02_model_comparison.png', 'fig_03_e3_leakage_ablation.png', 'fig_04_conformal_methods.png', 'fig_05_coverage_subgroups.png', 'fig_06_feature_importance.png')]
import importlib.metadata
runtime = {
    'commit': commit,
    'run_id': summary['run_id'],
    'python': sys.version,
    'packages': {dist.metadata['Name']: dist.version for dist in importlib.metadata.distributions() if dist.metadata['Name']},
}
runtime_path = Path('outputs/metrics/kaggle_runtime.json')
runtime_path.write_text(json.dumps(runtime, indent=2), encoding='utf-8')
files.append(runtime_path)
package = Path('/kaggle/working') / f"cs114_{summary['run_id']}.tar.gz"
with tarfile.open(package.with_suffix('.tmp'), 'w:gz') as tar:
    for path in files:
        tar.add(path, arcname=path.as_posix(), recursive=False)
package.with_suffix('.tmp').replace(package)
print(f'Tải kết quả tại {package}')

Lượt chạy thành công sẽ in đường dẫn gói kết quả chứa run ID.